In [1]:
import dgl.nn as dglnn
from dgl import from_networkx
import torch.nn as nn
import torch as th
import torch.nn.functional as F
import dgl.function as fn
import networkx as nx
import pandas as pd
import socket
import struct
import random
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import category_encoders as ce
from sklearn.decomposition import PCA
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np


In [2]:
data = pd.read_csv('ton_iot.csv')

In [10]:


data.type.value_counts()



NameError: name 'data' is not defined

In [4]:
data = data.sample(frac=0.1,random_state = 123)

In [5]:
data['src_ip'] = data.src_ip.apply(lambda x: socket.inet_ntoa(struct.pack('>I', random.randint(0xac100001, 0xac1f0001))))


In [6]:
data['src_ip'] = data.src_ip.apply(str)
data['src_port'] = data.src_port.apply(str)
data['dst_ip'] = data.dst_ip.apply(str)
data['dst_port'] = data.dst_port.apply(str)


In [7]:
data['src_ip'] = data['src_ip'] + ':' + data['src_port']
data['dst_ip'] = data['dst_ip'] + ':' + data['dst_port']


In [8]:
data['http_trans_depth'].unique()


array(['-', '1', '3', '2', '7'], dtype=object)

In [9]:
data.drop(columns=['ts','src_port','dst_port','http_uri','weird_name','weird_addl','weird_notice','dns_query','ssl_subject','ssl_issuer','http_user_agent','type'],inplace=True)


In [8]:
data

NameError: name 'data' is not defined

In [11]:


data['label'] = data.label.apply(int)

label = data.label



In [12]:
data.drop(columns=['label'],inplace = True)



In [13]:
scaler = StandardScaler()

data =  pd.concat([data, label], axis=1)



In [14]:
X_train, X_test, y_train, y_test = train_test_split(
     data, label, test_size=0.7, random_state=123,stratify= label)


In [15]:
encoder = ce.TargetEncoder(cols=['proto','service','conn_state','dns_qclass','dns_qtype','dns_rcode','dns_AA','dns_RD','dns_RA','dns_rejected','ssl_version','ssl_cipher','ssl_resumed','ssl_established','http_method','http_version','http_status_code','http_orig_mime_types','http_resp_mime_types','http_trans_depth'])
encoder.fit(X_train, y_train)
X_train = encoder.transform(X_train)


In [16]:
cols_to_norm = list(set(list(X_train.iloc[:, 2:].columns ))  - set(list(['label'])) )
X_train[cols_to_norm] = scaler.fit_transform(X_train[cols_to_norm])


In [17]:
X_train['h'] = X_train[ cols_to_norm ].values.tolist()


In [18]:
G = nx.from_pandas_edgelist(X_train, "src_ip", "dst_ip", ['h','label'],create_using=nx.MultiGraph())


In [19]:
G = G.to_directed()

G = from_networkx(G,edge_attrs=['h','label'] )



In [20]:
G.ndata['h'] = th.ones(G.num_nodes(), G.edata['h'].shape[1])


In [21]:
G.ndata['h'] = th.reshape(G.ndata['h'], (G.ndata['h'].shape[0], 1,G.ndata['h'].shape[1]))
G.edata['h'] = th.reshape(G.edata['h'], (G.edata['h'].shape[0], 1,G.edata['h'].shape[1]))


In [22]:
G.edata['train_mask'] = th.ones(len(G.edata['h']), dtype=th.bool)


In [23]:
class Model(nn.Module):
    def __init__(self, ndim_in, ndim_out, edim, activation, dropout):
        super().__init__()
        self.gnn = SAGE(ndim_in, ndim_out, edim, activation, dropout)
        self.pred = MLPPredictor(ndim_out, 10)
    def forward(self, g, nfeats, efeats):
        h = self.gnn(g, nfeats, efeats)
        return self.pred(g, h)



In [24]:
class SAGELayer(nn.Module):
    def __init__(self, ndim_in, edims, ndim_out, activation):
        super(SAGELayer, self).__init__()
        ### force to outut fix dimensions
        self.W_msg = nn.Linear(ndim_in + edims, ndim_out)
        ### apply weight
        self.W_apply = nn.Linear(ndim_in + ndim_out, ndim_out)
        self.activation = activation

    def message_func(self, edges):
        return {'m': self.W_msg(th.cat([edges.src['h'], edges.data['h']], 2))}

    def forward(self, g_dgl, nfeats, efeats):
        with g_dgl.local_scope():
            g = g_dgl
            g.ndata['h'] = nfeats
            g.edata['h'] = efeats
            # Eq4
            g.update_all(self.message_func, fn.mean('m', 'h_neigh'))
            # Eq5          
            g.ndata['h'] = F.relu(self.W_apply(th.cat([g.ndata['h'], g.ndata['h_neigh']], 2)))
            return g.ndata['h']


class SAGE(nn.Module):
    def __init__(self, ndim_in, ndim_out, edim, activation, dropout):
        super(SAGE, self).__init__()
        self.layers = nn.ModuleList()
        self.layers.append(SAGELayer(ndim_in, edim, 128, activation))
        self.layers.append(SAGELayer(128, edim, ndim_out, activation))
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, g, nfeats, efeats):
        for i, layer in enumerate(self.layers):
            if i != 0:
                nfeats = self.dropout(nfeats)
            nfeats = layer(g, nfeats, efeats)
        return nfeats.sum(1)


In [25]:
class MLPPredictor(nn.Module):
    def __init__(self, in_features, out_classes):
        super().__init__()
        self.W = nn.Linear(in_features * 2, out_classes)

    def apply_edges(self, edges):
        h_u = edges.src['h']
        h_v = edges.dst['h']
        global score
        global emb
        emb = th.cat([h_u, h_v], 1)
        score = self.W(th.cat([h_u, h_v], 1))
        return {'score': score}

    def forward(self, graph, h):
        with graph.local_scope():
            graph.ndata['h'] = h
            graph.apply_edges(self.apply_edges)
            return graph.edata['score']


In [26]:
G.device

device(type='cpu')

In [27]:
G.ndata['h'].device


device(type='cpu')

In [28]:
G.edata['h'].device

device(type='cpu')

In [29]:
from sklearn.utils import class_weight
class_weights = class_weight.compute_class_weight('balanced',
                                                 classes=np.unique(G.edata['label'].cpu().numpy()),
                                                 y=G.edata['label'].cpu().numpy())

In [30]:
class_weights = th.FloatTensor(class_weights)#.cuda()
criterion = nn.CrossEntropyLoss(weight = class_weights)

In [31]:
def compute_accuracy(pred, labels):
    return (pred.argmax(1) == labels).float().mean().item()

In [32]:
import os
node_features = G.ndata['h']
edge_features = G.edata['h']

edge_label = G.edata['label']
train_mask = G.edata['train_mask']

model = Model(G.ndata['h'].shape[2], 128, G.ndata['h'].shape[2], F.relu, 0.2)#.cuda()
opt = th.optim.Adam(model.parameters())

In [6]:
G

NameError: name 'G' is not defined

In [34]:
import os
os.remove("cluster_gcn.pkl")

In [35]:
import dgl
import dgl.nn as dglnn
import torch

In [36]:
num_parts = 1000
sampler = dgl.dataloading.ClusterGCNSampler(G, num_parts, prefetch_edata=["h", "label", "train_mask"]#, "val_mask", "test_mask"]
)

Convert a graph into a bidirected graph: 0.123 seconds, peak memory: 32.332 GB
Construct multi-constraint weights: 0.019 seconds, peak memory: 32.332 GB
Metis partitioning: 1.189 seconds, peak memory: 32.332 GB


[11:00:34] /opt/dgl/src/graph/transform/metis_partition_hetero.cc:89: Partition a graph with 775908 nodes and 1318720 edges into 1000 parts and get 319255 edge cuts


In [37]:
dataloader = dgl.dataloading.DataLoader(
    G, torch.arange(num_parts), sampler,
    batch_size=20, shuffle=True, drop_last=False, num_workers=4)

In [4]:
import time
for _ in range(10):
    t0 = time.time()
    history = model.train()
    for it, sg in enumerate(dataloader):
        x = sg.edata["h"]
        x2 = sg.ndata["h"]
        y = sg.edata["label"]
        m = sg.edata["train_mask"].bool()
        y_hat = model(sg, x2, x)
        loss = F.cross_entropy(y_hat[m], y[m])
        opt.zero_grad()
        loss.backward()
        opt.step()
        if it % 20 == 0:
            #acc = MF.accuracy(y_hat[m], y[m])
            #mem = torch.cuda.max_memory_allocated() / 1000000
            #print("Loss", loss.item(), "Acc", acc.item(), "GPU Mem", mem, "MB")
            print('Epoch:', it ,' Training acc:', compute_accuracy(y_hat[m], y[m]))
           # history = model.fit(y_hat[m], y[m], epochs=5, batch_size=1)
    tt = time.time()
    print(tt - t0)
    #durations.append(tt - t0)

NameError: name 'model' is not defined

In [39]:
X_test = encoder.transform(X_test)

X_test[cols_to_norm] = scaler.transform(X_test[cols_to_norm])



In [40]:
X_test['h'] = X_test[ cols_to_norm ].values.tolist()


In [41]:
G_test = nx.from_pandas_edgelist(X_test, "src_ip", "dst_ip", ['h','label'],create_using=nx.MultiGraph())
G_test = G_test.to_directed()
G_test = from_networkx(G_test,edge_attrs=['h','label'] )
#actual = G_test.edata.pop('label')
G_test.ndata['feature'] = th.ones(G_test.num_nodes(),G.ndata['h'].shape[2])

In [42]:
G_test.ndata['feature'] = th.reshape(G_test.ndata['feature'], (G_test.ndata['feature'].shape[0], 1, G_test.ndata['feature'].shape[1]))
G_test.edata['h'] = th.reshape(G_test.edata['h'], (G_test.edata['h'].shape[0], 1, G_test.edata['h'].shape[1]))



In [43]:
G_test.edata['test_mask'] = th.ones(len(G_test.edata['h']), dtype=th.bool)

In [44]:
import os
os.remove("cluster_gcn.pkl")

In [45]:
num_parts = 2000
sampler = dgl.dataloading.ClusterGCNSampler(G_test, num_parts, prefetch_edata=["h", "label", "test_mask"]#, "val_mask", "test_mask"]
)

Convert a graph into a bidirected graph: 0.456 seconds, peak memory: 32.332 GB
Construct multi-constraint weights: 0.001 seconds, peak memory: 32.332 GB
Metis partitioning: 3.735 seconds, peak memory: 32.332 GB


[11:06:35] /opt/dgl/src/graph/transform/metis_partition_hetero.cc:89: Partition a graph with 1737818 nodes and 3077044 edges into 2000 parts and get 782796 edge cuts


In [46]:
# importing the module
#import tracemalloc
 
# code or function for which memory
# has to be monitored
#def app():
#    lt = []
#    num_parts = 2000
#    sampler = dgl.dataloading.ClusterGCNSampler(G_test, num_parts, prefetch_edata=["h", "label", "test_mask"]#, "val_mask", "test_mask"]
#    )
#    for i in range(0, 100000):
#        lt.append(i)
 
# starting the monitoring
#tracemalloc.start()
 
# function call
#app()
 
# displaying the memory
#print(tracemalloc.get_traced_memory())
 
# stopping the library
#tracemalloc.stop()

In [47]:
dataloader = dgl.dataloading.DataLoader(
    G_test, torch.arange(num_parts), sampler,
    batch_size=20, shuffle=True, drop_last=False, num_workers=4)

In [48]:
model.eval()

Model(
  (gnn): SAGE(
    (layers): ModuleList(
      (0): SAGELayer(
        (W_msg): Linear(in_features=60, out_features=128, bias=True)
        (W_apply): Linear(in_features=158, out_features=128, bias=True)
      )
      (1): SAGELayer(
        (W_msg): Linear(in_features=158, out_features=128, bias=True)
        (W_apply): Linear(in_features=256, out_features=128, bias=True)
      )
    )
    (dropout): Dropout(p=0.2, inplace=False)
  )
  (pred): MLPPredictor(
    (W): Linear(in_features=256, out_features=2, bias=True)
  )
)

In [49]:
import torchmetrics.functional as MF

In [50]:
import timeit
totaltime=0
start_time = timeit.default_timer()
with torch.no_grad():
        val_preds, test_preds = [], []
        val_labels, test_labels = [], []
        for it, sg in enumerate(dataloader):
            x = sg.edata["h"]
            x2 = sg.ndata["feature"]
            y = sg.edata["label"]
            #m_val = sg.ndata["val_mask"].bool()
            m_test = sg.edata["test_mask"].bool()
            start_time = timeit.default_timer()
            y_hat = model(sg, x2, x)
            elapsed = timeit.default_timer() - start_time
            totaltime=totaltime+elapsed
           #val_preds.append(y_hat[m_val])
            #val_labels.append(y[m_val])
            test_preds.append(y_hat[m_test])
            test_labels.append(y[m_test])
            #break
       # val_preds = torch.cat(val_preds, 0)
       # val_labels = torch.cat(val_labels, 0)
        test_preds = torch.cat(test_preds, 0)
        test_labels = torch.cat(test_labels, 0)

        test_acc = MF.accuracy(
            test_preds,
            test_labels,
            task="multiclass",
           num_classes=2,
        )
        print("Test acc:", test_acc.item())
        #print("Test acc:", compute_accuracy(y_hat[m_test], y[m_test]))
        #tt = time.time()
        #print(tt - t0)ton_iot

/home/tselab/anaconda3/lib/python3.10/site-packages/dgl/dataloading/dataloader.py:1043: DGLWarning: Dataloader CPU affinity opt is not enabled, consider switching it on (see enable_cpu_affinity() or CPU best practices for DGL [https://docs.dgl.ai/tutorials/cpu/cpu_best_practises.html])
  dgl_warning(


Test acc: 0.9772076606750488


In [51]:
print(str(elapsed) + ' seconds')

0.03933771303854883 seconds


In [52]:
print(str(totaltime) + ' seconds')

4.9414257350726984 seconds


In [53]:
emb_viz=test_preds

In [54]:
df_umap = pd.DataFrame(emb_viz, columns=['comp1', 'comp2'])

In [55]:
test_preds = test_preds.argmax(1)
test_preds = th.Tensor.cpu(test_preds).detach().numpy()

In [1]:
from sklearn.metrics import classification_report
#target_names = ['Normal', 'Abnormal']
print(classification_report(test_labels, test_preds, digits=4)#,target_names=target_names))

SyntaxError: incomplete input (1398856754.py, line 3)